In [ ]:
pip install llama-index llama-index-llms-openai python-dotenv langchain-community langchain-openai openai langchain-text-splitters tiktoken colorama langsmith zod nltk

Install Langsmith SDK

In [ ]:
pip install -U openai langsmith

## Setup

**Loading Environment Variables**

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
from colorama import Fore, Style
import os

load_dotenv(override=True)  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded from environment variable.{Style.RESET_ALL}")

**LLM Setup and OpenAI Secret Key**

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

embed_model = OpenAIEmbedding(
    api_key=os.getenv("OPENAI_API_KEY"), model="text-embedding-3-small"
)

llm = OpenAI(model="gpt-4o-mini", temperature=0)

## Loading and Chunking Documents

In [ ]:
# -----------------------------
# Download Data
# -----------------------------

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/refs/heads/main/docs/examples/data/paul_graham/paul_graham_essay_ambiguated.txt' -O 'data/paul_graham/paul_graham_essay-ambiguated.txt'

In [ ]:
from llama_index.core import SimpleDirectoryReader


documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

helper functions : count tokens, split or summarize data source

In [ ]:
import tiktoken
from openai import OpenAI

client = OpenAI() 
MODEL = "gpt-4o-mini"
MAX_INPUT_TOKENS = 6000   # safe margin under 8k

def count_tokens(text: str, model: str = MODEL) -> int:
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

def split_by_tokens(text: str, max_tokens: int = 3000):
    enc = tiktoken.encoding_for_model(MODEL)
    tokens = enc.encode(text)

    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunks.append(enc.decode(chunk_tokens))

    return chunks

def split_text(text: str, max_chars: int = 4000):
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]

def summarize(text: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a precise summarizer. Keep only key facts."
            },
            {
                "role": "user",
                "content": f"Summarize this:\n\n{text}"
            }
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

def preprocess_documents(documents):
    processed_docs = []

    for doc in documents:
        text = doc.text if hasattr(doc, "text") else str(doc)
        tokens = count_tokens(text)

        # ✅ If small → keep as is
        if tokens <= MAX_INPUT_TOKENS:
            processed_docs.append(doc)
            continue

        # ❗If too large → summarize in chunks (map-reduce)
        chunks = split_by_tokens(text, max_tokens=3000)

        chunk_summaries = [summarize(chunk) for chunk in chunks]

        # final reduce step
        final_summary = summarize("\n".join(chunk_summaries))

        # replace content
        doc.text = final_summary
        processed_docs.append(doc)

    return processed_docs

**Text/Sentence Chunking**

In [ ]:
from llama_index.core.node_parser import (
    SentenceSplitter,
    SemanticSplitterNodeParser,
)

sentence_splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=75
)

**Semantic Chunking**
- [Semantic Splitter](https://developers.llamaindex.ai/python/examples/node_parsers/semantic_chunking/)
- https://medium.com/@muhanad.a.husn/semantic-chunking-using-semanticsplitternodeparser-7b5b525e27fe

In this example, buffer_size and breakpoint_percentile_threshold are parameters we can tune.  With this setup, the splitter will iterate through the document, compute embeddings, compare similarities, and split the text at points where the content’s semantic similarity dips below the chosen threshold.

In [ ]:
# Semantic splitter initialization

from llama_index.core.node_parser import SemanticSplitterNodeParser
import nltk
nltk.download('punkt_tab')

semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=2,
    breakpoint_percentile_threshold=75,
    embed_model=embed_model,
    sentence_splitter=lambda text: nltk.sent_tokenize(text),  # explicit sentence boundary
)

In [ ]:
import nltk
from llama_index.core import Document

nodes = sentence_splitter.get_nodes_from_documents(documents)
for i, node in enumerate(nodes[:2]):  # just first 2
    embedding = embed_model.get_text_embedding(node.text)
    
    print(f"\n📄 Node {i}")
    print("Text:", node.text[:100])
    print("Embedding (first 10 values):", embedding[:10])


semantic_nodes = semantic_splitter.build_semantic_nodes_from_documents(nodes)
print(f"Total semantic nodes: {len(semantic_nodes)}")

for i, node in enumerate(semantic_nodes[:3]):
    text = node.text
    # Verify no mid-sentence cuts: check start/end characters
    print(f"\n📄 Node {i}")
    print(f"  Starts: ...{text[:80]}")
    print(f"  Ends:   ...{text[-80:]}")
    print(f"  Length: {len(text)} chars")
    print(f"  Ends with period/punctuation: {text.rstrip()[-1] in '.!?'}")


## Build Index and retriever

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding()

index = VectorStoreIndex.from_documents(
    documents,
    transformations=[sentence_splitter, semantic_splitter],
    embed_model=embed_model
)

retriever = index.as_retriever(similarity_top_k=10)

# Context Optimization Strategies

1. Summarization Compression
- *when context too large (based on token count) → summarize*

In [ ]:
from llama_index.llms.openai import OpenAI
from langsmith import traceable

llm = OpenAI(model="gpt-4o-mini", temperature=0)

@traceable()
def summarize_context(nodes):
    combined_text = "\n\n".join([n.text for n in nodes])

    prompt = f"""
Summarize the following context while preserving key facts:

{combined_text}

Summary:
"""
    response = llm.complete(prompt)
    return str(response)

2. Relevance-based Truncation
- *limit context based on token count*

In [ ]:
import tiktoken

@traceable()
def count_tokens(text):
    enc = tiktoken.encoding_for_model("gpt-4o-mini")
    return len(enc.encode(text))

@traceable()
def truncate_by_tokens(nodes, max_tokens=1500):
    selected = []
    total_tokens = 0

    for node in nodes:
        node_text = node.text
        tokens = count_tokens(node_text)

        if total_tokens + tokens > max_tokens:
            break

        selected.append(node)
        total_tokens += tokens

    return selected

## Build the Pipeline (Optimized)

In [ ]:


@traceable()
def optimized_rag(query):
    print(f"\n🔍 Query: {query}")

    # 1. Retrieve
    nodes = retriever.retrieve(query)
    print(f"\n📄 Retrieved {len(nodes)} chunks")

    # 2. Truncate by token budget
    filtered_nodes = truncate_by_tokens(nodes, max_tokens=1200)
    print(f"✂️ After truncation: {len(filtered_nodes)} chunks")

    # 3. Optional summarization (if still large)
    total_tokens = sum(count_tokens(n.text) for n in filtered_nodes)

    if total_tokens > 1000:
        print("🧠 Applying summarization...")
        context = summarize_context(filtered_nodes)
    else:
        
        # optimized context
        context = "\n\n".join([n.text for n in filtered_nodes])

    # 4. Final prompt
    prompt = f"""
        Answer the question using ONLY the context below.

        Context:
        {context}

        Question: {query}
        Answer:
        """

    response = llm.complete(prompt)

    return response

@traceable()
def main(query):
    print(f"\n🔍 Query: {query}")

    # 1. Retrieve
    nodes = retriever.retrieve(query)
   
    context = "\n\n".join([n.text for n in nodes])

    # 4. Final prompt
    prompt = f"""
        Answer the question using ONLY the context below.

        Context:
        {context}

        Question: {query}
        Answer:
        """
    return llm.complete(prompt)

# Run the Pipeline
* testing the **basic** and **optimized** pipelines.
* tracing available on the [LangSmith](https://smith.langchain.com/) dashboard.

In [ ]:

response = main("Who is Paul Graham and what did he work on?")
print("\n💡 Answer:\n", response)

response = optimized_rag("Who is Paul Graham and what did he work on?")
print("\n💡 Answer:\n", response)